In [ ]:
import argparse
from transformers import AutoTokenizer, AutoModel
from classifier.model import ClassModel
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import torch
import os
import itertools
from tqdm import tqdm
import datasets
import json
import pickle
from sklearn.metrics.pairwise import cosine_similarity as cos
from collections import defaultdict

In [ ]:
data_dir = './LitSearch'

In [ ]:
id2label, label2id, id2name, name2id = {}, {}, [], {}
with open('classifier/labels.txt') as f:
    for i, line in enumerate(f):
        label, name, _ = line.strip().split('\t')
        id2label[i] = label
        label2id[label] = i
        id2name.append(name)
        name2id[name] = i
corpus_with_labels = json.load(open(f'{data_dir}/specter2_corpus_with-topic-terms.json'))
corpus_embeddings = torch.load(f'{data_dir}/corpus-enc-specter.pt')
id2corpus_id = pickle.load(open(f'{data_dir}/corpus-enc-index.pkl', 'rb'))
raw_queries = datasets.load_dataset("princeton-nlp/LitSearch", "query", split="full")

In [ ]:
(id2phrase, phrase2id) = pickle.load(open(f'{data_dir}/specter2_corpus_with-topic-terms.json.phrase_idx.pkl', 'rb'))
phrase_embeddings = torch.load(f'{data_dir}/specter2_corpus_with-topic-terms.json.phrase-enc-specter.pt')
topic_embeddings = torch.load(f'{data_dir}/topic-enc-specter.pt')

In [ ]:
gpu_num = 0
backbone = 'allenai/specter2_base'
tokenizer = AutoTokenizer.from_pretrained(backbone)
encoder = AutoModel.from_pretrained(backbone).to(f'cuda:{gpu_num}')

In [ ]:
all_embeddings = torch.cat((topic_embeddings, phrase_embeddings), dim=0).numpy()
all_embeddings = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)
all_terms = id2name + id2phrase
all_term2id = {t:i for i,t in enumerate(all_terms)}
doc2term_id = {}
for corpusid in corpus_with_labels:
    doc_terms = [t[1] for t in corpus_with_labels[corpusid]['topics']] +\
                       [t.lower() for t in corpus_with_labels[corpusid]['terms']]
    doc2term_id[corpusid] = set([all_term2id[t] for t in doc_terms if t != ''])

In [ ]:
import openai
from openai import OpenAI
import os
import time

os.environ["OPENAI_API_KEY"] = 'your key'
client = OpenAI()

def api_call(doc, instruction=None, temperature=0.2, model='gpt-4o-mini'):
    '''
    doc str: query
    instruction str: system instruction
    demos List((str, str)): demonstrations, if any
    temperature: None for default temprature
    '''

    if instruction:
        messages = [{"role": "system", "content": instruction}]
    else:
        messages = []

    messages.append({"role": "user", "content": doc})

    timeout_num = 0
    while timeout_num < 3:
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
            )
            break
        except openai.APITimeoutError:
            print('Timeout!')
            timeout_num += 1
            continue
        except openai.RateLimitError:
            print('RateLimitError')
            time.sleep(60)
            continue

    if timeout_num >= 3:
        return ''

    return completion.choices[0].message.content

In [ ]:
def evaluate_metrics(results, ks=[1, 5]):
    """
    Evaluate Hit@k, Recall@k, and Precision@k for the given ks.
    
    Parameters:
    - jsonl_file_path: Path to the JSONL file containing queries, ground truths, and retrieved documents.
    - ks: List of integer values for k to compute metrics at.
    
    Returns:
    - metrics: A dictionary containing average metrics for each k.
    """
    # Initialize accumulators for each metric and k
    total_hits_at_k = {k: 0.0 for k in ks}
    total_recall_at_k = {k: 0.0 for k in ks}
    total_precision_at_k = {k: 0.0 for k in ks}
    num_queries = 0

    for r in results:
        ground_truth = set(r['corpusids'])  # Ground truth relevant documents
        retrieved = [i for i,_ in r['retrieved']]          # Retrieved documents

        num_relevant = len(ground_truth)
        if num_relevant == 0:
            # Skip queries with no ground truth
            continue

        num_queries += 1

        for k in ks:
            retrieved_at_k = retrieved[:k]
            retrieved_set = set(retrieved_at_k)

            # Compute Hit@k
            hit = 1.0 if len(ground_truth & retrieved_set) > 0 else 0.0
            total_hits_at_k[k] += hit

            # Compute Recall@k
            num_relevant_at_k = len(ground_truth & retrieved_set)
            recall_at_k = num_relevant_at_k / num_relevant
            total_recall_at_k[k] += recall_at_k


    # Compute average metrics
    average_hits_at_k = {k: total_hits_at_k[k] / num_queries if num_queries > 0 else 0.0 for k in ks}
    average_recall_at_k = {k: total_recall_at_k[k] / num_queries if num_queries > 0 else 0.0 for k in ks}

    # Print results
    print(f"Total Queries Evaluated: {num_queries}")
    for k in ks:
        print(f"\nMetrics at {k}:")
        print(f"Hit@{k}: {average_hits_at_k[k]:.4f}")
        print(f"Recall@{k}: {average_recall_at_k[k]:.4f}")

    # Return metrics if needed
    metrics = {
        'hit@k': average_hits_at_k,
        'recall@k': average_recall_at_k
    }
    return metrics

In [ ]:
query_instruction = '''
You will receive a query for computer science research papers and a ranked list of papers returned by a retriever.
You will also be provided a list of research topics and key terms with their frequencies that are frequently mentioned by the top-100 papers returned by the retriever.
Your task is to improve the provided retrieval results by selecting a list of topics and terms that can accurately identify the relevant papers of the query.
Make sure your selection is strictly based on the original query and does not contain repeated concepts.
Output format: '<ans>selection 1, selection 2, ...</ans>'.
Retriever result:
%s
Candidate topics:
%s
Candidate key terms:
%s
Original Query:
%s
'''

In [ ]:
def score_mapping(s, scores):
    return (s-np.mean(scores))/np.std(scores)

results = []
for query_id, ori_q in enumerate(tqdm(raw_queries)):
    query_text = ori_q['query']
    
    # base retriever
    inputs = tokenizer(query_text, return_tensors="pt", truncation=True, max_length=512).to(f'cuda:{gpu_num}')
    with torch.no_grad():
        output = encoder(**inputs)
    emb = output.last_hidden_state[0, 1:-1].mean(dim=0).cpu().numpy()
    
    scores = cos(emb[np.newaxis, :], corpus_embeddings)[0]
    top_k = np.argsort(-scores)[:1000]
    curr_results = [(id2corpus_id[i], scores[i]) for i in top_k]
    scores = np.array([s for _,s in curr_results])
    curr_results = [(i, score_mapping(s, scores)) for i,s in curr_results]
    
    # construct candidate concepts
    topk_terms_dict = defaultdict(int)
    topk_topics_dict = defaultdict(int)
    N = 100
    Nk = 50
    for rank, (corpusid, score) in enumerate(curr_results[:N]):
        for _, topic, _ in corpus_with_labels[str(corpusid)]['topics']:
            topk_topics_dict[topic] += 1
        for term in corpus_with_labels[str(corpusid)]['terms']:
            topk_terms_dict[term.lower()] += 1
    top_terms = [f'{t}, {c}' for t,c in sorted(topk_terms_dict.items(), key=lambda x: x[1], reverse=True)[:Nk]]
    top_topics = [f'{t}, {c}' for t,c in sorted(topk_topics_dict.items(), key=lambda x: x[1], reverse=True)[:Nk]]
    top_papers = ['[{}] Title: {}\nAbstract: {}'.format(rank+1, corpus_with_labels[str(corpusid)]['title'],\
                                                  corpus_with_labels[str(corpusid)]['abstract']) for rank, (corpusid, _) in enumerate(curr_results[:Nk])]
    
    llm_input = query_instruction % ('\n'.join(top_papers), '\n'.join(top_topics), '\n'.join(top_terms), query_text)
    llm_output = api_call(llm_input, model='gpt-4.1-mini')

    selected_concepts = []
    if '<ans>' in llm_output:
        selected_concepts = [t.lower() for t in llm_output.split('<ans>')[1].split('</ans>')[0].split(', ')]
        selected_concepts = [t for t in selected_terms if t in all_term2id]
    if len(selected_concepts) == 0:
        res = {
            'query': query_text,
            'corpusids': ori_q['corpusids'],
            'retrieved': curr_results,
        }
        results.append(res)
        continue

    # Concept-Based Ranking
    all_term_emb = all_embeddings[[all_term2id[t] for t in selected_concepts]]
    new_results = []
    for corpusid, score in curr_results:
        doc_embeddings = all_embeddings[list(doc2term_id[str(corpusid)])]
        if len(doc_embeddings) == 0:
            new_results.append((corpusid, 0, score))
            continue
        new_score = np.matmul(all_term_emb, doc_embeddings.T).max(axis=1).mean()
        new_results.append((corpusid, new_score, score))
        
    all_scores = np.array([s for _,s, _ in new_results])
    new_results = [(i, c+score_mapping(s, all_scores)) for i,s,c in new_results]
    new_results = sorted(new_results, key=lambda x: x[1], reverse=True)
    
    res = {
        'query': query_text,
        'corpusids': ori_q['corpusids'],
        'retrieved': new_results,
    }
    results.append(res)

In [ ]:
e = evaluate_metrics(results, [5, 20, 100])